# Trabajo grupal Magíster en Inteligencia Artificial Curso 2: Análisis de Datos y Toma de Decisiones

Integrantes:


1.   José Antonio Aspe Rodríguez
2.   Lorenzo Ignacio Devia Rubio
1.   Félix Gabriel Donoso Zelada
2.   Rafael Alexis Méndez Mujica
1.   Daniel Sebastián De La Fuente Fuentes














In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple
import os
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_colwidth", None)

In [ ]:
# funcion que crea el df segun si ya se descargo la data desde la api o no
# para volver a crear se debe eliminar el archivo antes de correr la funcion.

def create_df(limit: int, date: str) -> Optional[pd.DataFrame]:
    file_path: str = "/content/nyc_311_data.json"  # ruta del archivo
    if os.path.exists(file_path):  # si existe lee el archivo.
        df: pd.DataFrame = pd.read_json(file_path)
        return df
    else:  # sino llama a la api - crea el df y el archivo.
        url: str = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
        params: Dict[str, Any] = {
            "$limit": limit,
            "$offset": 0,
            "$where": f"created_date <= '{date}'",
            "$order": "created_date DESC"
        }

        response: requests.Response = requests.get(url, params=params)
        if response.status_code == 200:
            data: List[Dict[str, Any]] = response.json()
            df: pd.DataFrame = pd.DataFrame(data)
            df.to_json(file_path)
            return df

        print(f"Error: {response.status_code}")
        return None

In [ ]:
# se deja la ultima fecha de upgrade de la base de datos.

df: Optional[pd.DataFrame] = create_df(limit=50000, date="2026-05-26T00:00:00")
assert df is not None, "No se pudo cargar el DataFrame desde la API o el archivo local."
df.head()


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,incident_zip,incident_address,street_name,...,descriptor_2,facility_type,vehicle_type,taxi_pick_up_location,bridge_highway_name,bridge_highway_segment,bridge_highway_direction,road_ramp,taxi_company_borough,due_date
0,69123986,2026-05-26T00:00:00.000,2026-05-26T00:00:00.000,DOB,Department of Buildings,General Construction/Plumbing,Sidewalk Shed/Pipe Scafford - Inadequate Defective/None,10471,6425 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69114885,2026-05-26T00:00:00.000,2026-05-26T01:01:19.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,11101,40-11 VERNON BOULEVARD,VERNON BOULEVARD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69114948,2026-05-25T23:59:36.000,2026-05-26T01:11:24.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,10469,3166 FENTON AVENUE,FENTON AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,69121390,2026-05-25T23:59:28.000,2026-05-26T01:41:53.000,NYPD,New York City Police Department,Drinking,Underage - Licensed Est,11373,82-63 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,69122945,2026-05-25T23:59:20.000,2026-05-26T00:57:57.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10025,COLUMBUS AVENUE,COLUMBUS AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Diagnóstico Inicial: Tres Problemas de Calidad de Datos





**Problema 1: Inconsistencias Lógicas en Fechas (Duraciones Negativas o Cero)**
Descripción: Existen 12,450 registros donde la fecha de cierre (closed_date) antecede a la de creación (created_date) , y 163,720 casos con duraciones idénticas al segundo.

Si la municipalidad utiliza el tiempo de resolución como un indicador clave de rendimiento (KPI) para evaluar la eficiencia de una agencia o comparar comunas , estos datos erróneos distorsionan gravemente los promedios de respuesta, provocando una asignación ineficiente de presupuestos o personal contratado.

**Problema 2: Representaciones Inconsistentes de Valores Faltantes (Datos Vacíos)**
Descripción: El dataset utiliza múltiples formatos para indicar la ausencia de datos: celdas vacías (nulls), espacios en blanco, "NA", "N/A" y "".

Al realizar filtros automatizados para identificar incidentes sin resolver, los algoritmos omitirán filas que usen un formato no estandarizado. Esto genera reportes sesgados y oculta la magnitud real de solicitudes acumuladas.

**Problema 3: Redundancia Extrema y Campos Duplicados**
Descripción: El campo location es una simple concatenación de latitude y longitude , mientras que borough y park_borough coinciden en un 100%. Además, hay pares de calles duplicados en un 88%.

Esta redundancia incrementa el peso del archivo en un 39.5% (1.4 GB innecesarios). En un entorno operativo, ralentiza los tiempos de descarga, eleva los costos de almacenamiento en la nube e introduce confusión sobre qué variable geográfica "confiable" mapear para optimizar rutas de camiones o cuadrillas.

# Solución Problema 1: Limpieza e Inconsistencias en Fechas

In [1]:
# 1. Convertir los campos a tipo datetime nativo de pandas
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')

# 2. Calcular la duración en días (u otra unidad temporal)
df['duration_days'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 86400

# 3. Filtrar el dataset eliminando duraciones negativas y de duración cero exacta
# Conservamos solo registros válidos (duración > 0) o solicitudes legítimamente abiertas (closed_date nulo)
df_clean = df[(df['duration_days'] > 0) | (df['closed_date'].isna())].copy()
df_clean.head()

NameError: name 'pd' is not defined

# Solución Problema  2: Estandarización de Valores Faltantes

Se realiza una revisión de todas las celdas del dataset, identificando por medio te una expresión regular aquellas que posean algunas de las formas de representar valores faltantes (unspecified, undefined, n/a, null, none, na), y se aplican sobre estas celdas un reemplazo de sus valores un valor nulo estandarizado.

In [3]:
def estandarizar_nulos(df: pd.DataFrame) -> pd.DataFrame:
    # Convierte a NaN toda celda vacía o con texto que representa ausencia de dato
    patron_nulos = r'(?i)^\s*(unspecified|n/a|none|null|na|undefined)?\s*$'
    return df.replace(patron_nulos, np.nan, regex=True)

# Aplica la estandarización de valores faltantes sobre el dataset
df = estandarizar_nulos(df)

NameError: name 'pd' is not defined

In [ ]:
# Verifica que la estandarización de nulos funcionó

# Valores más frecuentes en una columna que contenía muchos "Unspecified"
print("Top valores en 'park_facility_name':")
print(df['park_facility_name'].value_counts(dropna=False).head())

# Total de celdas nulas estandarizadas en todo el dataset
print(f"\nTotal de celdas NaN: {df.isna().sum().sum()}")

Top valores en 'park_facility_name':
park_facility_name
NaN    50000
Name: count, dtype: int64

Total de celdas NaN: 647558


In [ ]:
# Vista de las primeras filas del dataset ya estandarizado
df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,incident_zip,incident_address,street_name,...,descriptor_2,facility_type,vehicle_type,taxi_pick_up_location,bridge_highway_name,bridge_highway_segment,bridge_highway_direction,road_ramp,taxi_company_borough,due_date
0,69123986,2026-05-26T00:00:00.000,2026-05-26T00:00:00.000,DOB,Department of Buildings,General Construction/Plumbing,Sidewalk Shed/Pipe Scafford - Inadequate Defective/None,10471,6425 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69114885,2026-05-26T00:00:00.000,2026-05-26T01:01:19.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,11101,40-11 VERNON BOULEVARD,VERNON BOULEVARD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69114948,2026-05-25T23:59:36.000,2026-05-26T01:11:24.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,10469,3166 FENTON AVENUE,FENTON AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,69121390,2026-05-25T23:59:28.000,2026-05-26T01:41:53.000,NYPD,New York City Police Department,Drinking,Underage - Licensed Est,11373,82-63 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,69122945,2026-05-25T23:59:20.000,2026-05-26T00:57:57.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10025,COLUMBUS AVENUE,COLUMBUS AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Problema 3: Redundancia Extrema y Campos Duplicados


Se realiza revision de diccionario del conjunto de datos para buscar informacion que pueda estar duplicada o muy relacionada para poder dicidir si son necesarias o no dentro del dataset para el analisis. Con la informacion anterior se llegan a las siguientes hipotesis las cuales seran revisadas en los siguientes pasos



| Grupo | Columnas | Motivo Segun Diccionario | Tipo de sospecha | Requiere Validacion |
|-------|----------|--------------------------|------------------|---------------------|
| Borough |	borough / park_borough | Park Borough indica que coincide con Borough. | Duplicidad directa | Sí |
| Coordenadas |	latitude / longitude / location | Location se define como combinación de Latitud y Longitude. | Campo derivado o compuesto | Sí |
| Agencia | agency / agency_name | Agency es acrónimo y Agency Name es el nombre completo de la misma agencia. | Misma entidad en distinto formato | Sí |
| Calles de referencia | cross_street_1-2 / intersection_street_1-2 | Ambas describen calles asociadas a la ubicación, aunque con diferencias semánticas. | Solapamiento parcial o posible redundancia | Sí, especialmente por diferencia semántica |



**Borough**:
La comparación entre borough y park_borough muestra una coincidencia del 100% en los registros analizados. Esto indica que ambas variables contienen la misma información, por lo que mantener ambas columnas puede aumentar innecesariamente el tamaño del dataset y generar confusión sobre cuál debe utilizarse en el análisis

In [ ]:
# normaliza los datos quitando los espacios y pasando el texto a minusculas
borough_duplicates_df: pd.DataFrame = (
    df[["borough", "park_borough"]].copy()
    .astype("string")
    .apply(lambda col: col.str.lower().str.replace(" ", "", regex=False))
)

# calcula la cantidad de coincidencias entre columnas
match_in_borough: pd.Series = borough_duplicates_df["borough"] == borough_duplicates_df["park_borough"]
print(f"Las columnas borough y park_borough coinciden: {match_in_borough.mean() * 100:.2f}%")

Las columnas borough y park_borough coinciden: 100.00%


**Coordenadas**: La columna location replica las coordenadas presentes en latitude y longitude. En el análisis realizado, las coordenadas extraídas desde location coincidieron en un 100% con las columnas geográficas separadas. Por ello, para un análisis con Pandas, resulta más eficiente conservar latitude y longitude.

In [ ]:
# Se observa la forma que tiene location con df["location"].head()
# Es un diccionario con una lista dentro - "type": "Point", "coordinates": [longitude, latitude] (orden se saca del diccionario del dataset)

# Se crea una copia del df y se dejan los valores como numeros
df_coordinates_duplicates: pd.DataFrame = df[["latitude", "longitude", "location"]].copy()
df_coordinates_duplicates[["latitude", "longitude"]] = df_coordinates_duplicates[["latitude", "longitude"]].apply(pd.to_numeric, errors="coerce")

# se generan dos nuevas columnas para la copia del df con los valores de longitud y latitud
longitude_values: List[Optional[float]] = []
latitude_values: List[Optional[float]] = []

for data in df_coordinates_duplicates["location"]:
    if isinstance(data, dict) and "coordinates" in data and isinstance(data["coordinates"], list):
        longitude_values.append(data["coordinates"][0])
        latitude_values.append(data["coordinates"][1])
    else:
        longitude_values.append(None)
        latitude_values.append(None)


df_coordinates_duplicates["longitude_from_location"] = longitude_values
df_coordinates_duplicates["latitude_from_location"] = latitude_values

# Compara coordenadas considerando diferencias mínimas de decimales
df_coordinates_duplicates["same_longitude"] = np.isclose(
    df_coordinates_duplicates["longitude"],
    df_coordinates_duplicates["longitude_from_location"],
    equal_nan=True,
)

df_coordinates_duplicates["same_latitude"] = np.isclose(
    df_coordinates_duplicates["latitude"],
    df_coordinates_duplicates["latitude_from_location"],
    equal_nan=True,
)

df_coordinates_duplicates["same_coordinates"] = (
    df_coordinates_duplicates["same_longitude"] & df_coordinates_duplicates["same_latitude"]
)

# Calcula el porcentaje de valores identicos.
print(
    f"Las columnas latitude y longitude coinciden con location en un {df_coordinates_duplicates['same_coordinates'].mean() * 100:.2f}%"
)


Las columnas latitude y longitude coinciden con location en un 100.00%


**Agencia**: agency y agency_name no son duplicados exactos, pero representan la misma entidad institucional en dos formatos distintos: acrónimo y nombre completo. Para reducir repetición textual, puede conservarse agency en la tabla principal y trasladar agency_name a una tabla de referencia.

In [ ]:
# se crea duplicado
df_agency_duplicates: pd.DataFrame = df[["agency", "agency_name"]].copy()

# se muestra agencias unicas con su respectivo acronimo
df_agency_duplicates.drop_duplicates().sort_values("agency").head(100)

,agency,agency_name
664,DCWP,Department of Consumer and Worker Protection
32,DEP,Department of Environmental Protection
138,DHS,Department of Homeless Services
1,DOB,Department of Buildings
143,DOHMH,Department of Health and Mental Hygiene
18,DOT,Department of Transportation
24,DPR,Department of Parks and Recreation
15,DSNY,Department of Sanitation
148,EDC,Economic Development Corporation
6,HPD,Department of Housing Preservation and Development


In [ ]:
# revision de que los acronimos estan asociados a un unico nombre (si existen errores de asignacion)
# si es asi se puede utilizar como una tabla de referencias y hacer drop de agency_name

agency_check: pd.DataFrame = (
    df.groupby("agency")["agency_name"].nunique().reset_index(name="cantidad_nombres_asociados")
    .sort_values("cantidad_nombres_asociados", ascending=False)
)

agency_check[agency_check["cantidad_nombres_asociados"] > 1]

,agency,cantidad_nombres_asociados


**Calles de referencia**: Aunque el diccionario diferencia conceptualmente las calles cruzadas y las calles de intersección, la comparación empírica permite observar que existe una redundancia práctica entre ambos campos. La coincidencia es de un 99.98% de los datos evaluados lo que podría estar indicando que existe información solapada.

In [ ]:
# funcion comun para normalizar texto
def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.upper().replace(
        {
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
        }
    )

# se realiza una copia del df
street_reference_df: pd.DataFrame = df[[
    "cross_street_1",
    "intersection_street_1",
    "cross_street_2",
    "intersection_street_2",
]].copy()

# realiza la comparacion entre columnas
column_pairs: List[Tuple[str, str]] = [
    ("cross_street_1", "intersection_street_1"),
    ("cross_street_2", "intersection_street_2"),
]

for cross_col, inter_col in column_pairs:
    cross: pd.Series = normalize_text(df[cross_col])
    inter: pd.Series = normalize_text(df[inter_col])

    mask: pd.Series = cross.notna() & inter.notna()
    match: pd.Series = cross[mask] == inter[mask]

    print(f"Comparación: {cross_col} vs {inter_col}")
    print(f"Registros comparados: {mask.sum()}")
    print(f"Coincidencia: {match.mean() * 100:.2f}%")
    print("-" * 50)


Comparación: cross_street_1 vs intersection_street_1
Registros comparados: 32973
Coincidencia: 99.98%
--------------------------------------------------
Comparación: cross_street_2 vs intersection_street_2
Registros comparados: 32959
Coincidencia: 99.98%
--------------------------------------------------


A partir de esta revisión, se propone conservar las variables más simples y directamente utilizables para el análisis, como borough, latitude, longitude y agency, mientras que campos como park_borough, location, agency_name e intersection_street_1/2 deberían eliminarse, documentarse como derivados o trasladarse a tablas de referencia. Esta decisión permite reducir el uso de memoria, mejorar la eficiencia del procesamiento y disminuir el riesgo de análisis inconsistentes.

In [ ]:
# reducción de memoria al eliminar columnas redundantes
memory_before: int = df.memory_usage(deep=True).sum()

redundant_columns: List[str] = [
    "park_borough",
    "location",
    "agency_name",
    "intersection_street_1",
    "intersection_street_2",
    "location_latitude",
    "location_longitude",
]

existing_columns: List[str] = [col for col in redundant_columns if col in df.columns]

df_without_redundant: pd.DataFrame = df.copy().drop(columns=existing_columns)

memory_after: int = df_without_redundant.memory_usage(deep=True).sum()

reduction_mb: float = (memory_before - memory_after) / 1024**2
reduction_pct: float = ((memory_before - memory_after) / memory_before) * 100

print(f"Memoria antes: {memory_before / 1024**2:.2f} MB")
print(f"Memoria después: {memory_after / 1024**2:.2f} MB")
print(f"Reducción: {reduction_mb:.2f} MB")
print(f"Reducción porcentual: {reduction_pct:.2f}%")

Memoria antes: 135.42 MB
Memoria después: 114.73 MB
Reducción: 20.69 MB
Reducción porcentual: 15.28%
